# 01 — Data Cleaning Base

This notebook loads, cleans, merges, validates, and exports the base data for the full Vanguard project.

In [ ]:

import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_DATA_PATH = "raw_data"
PROCESSED_DATA_PATH = "processed_data"

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)


## Load raw data

In [ ]:

demo = pd.read_csv(f"{RAW_DATA_PATH}/df_final_demo.csv")
experiment = pd.read_csv(f"{RAW_DATA_PATH}/df_final_experiment_clients.csv")
web_pt1 = pd.read_csv(f"{RAW_DATA_PATH}/df_final_web_data_pt_1.csv")
web_pt2 = pd.read_csv(f"{RAW_DATA_PATH}/df_final_web_data_pt_2.csv")

print("Demo:", demo.shape)
print("Experiment:", experiment.shape)
print("Web part 1:", web_pt1.shape)
print("Web part 2:", web_pt2.shape)


## Standardize column names

In [ ]:

def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

demo = clean_column_names(demo)
experiment = clean_column_names(experiment)
web_pt1 = clean_column_names(web_pt1)
web_pt2 = clean_column_names(web_pt2)


## Combine web data

In [ ]:

web = pd.concat([web_pt1, web_pt2], ignore_index=True)

print("Combined web shape:", web.shape)
web.head()


## Check missing values

In [ ]:

display(demo.isna().sum())
display(experiment.isna().sum())
display(web.isna().sum())


## Remove duplicates

In [ ]:

print("Duplicates before:")
print("Demo:", demo.duplicated().sum())
print("Experiment:", experiment.duplicated().sum())
print("Web:", web.duplicated().sum())

demo = demo.drop_duplicates()
experiment = experiment.drop_duplicates()
web = web.drop_duplicates()

print("Duplicates after:")
print("Demo:", demo.duplicated().sum())
print("Experiment:", experiment.duplicated().sum())
print("Web:", web.duplicated().sum())


## Clean experiment data

In [ ]:

experiment["variation"] = experiment["variation"].str.lower().str.strip()

display(experiment["variation"].value_counts(dropna=False))

experiment = experiment.dropna(subset=["variation"])
experiment = experiment[experiment["variation"].isin(["control", "test"])]

display(experiment["variation"].value_counts())


## Clean process steps

In [ ]:

web["process_step"] = web["process_step"].str.lower().str.strip()

step_mapping = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

web["step_num"] = web["process_step"].map(step_mapping)

display(web["process_step"].value_counts(dropna=False))
display(web[web["step_num"].isna()]["process_step"].value_counts())


## Convert date and time

In [ ]:

web["date_time"] = pd.to_datetime(web["date_time"], errors="coerce")

web["date"] = web["date_time"].dt.date
web["time"] = web["date_time"].dt.time
web["hour"] = web["date_time"].dt.hour
web["weekday"] = web["date_time"].dt.day_name()


## Clean demographic data

In [ ]:

demo = demo.drop_duplicates(subset=["client_id"])

print("Demo shape:", demo.shape)
display(demo.isna().sum())


## Merge datasets

In [ ]:

client_data = demo.merge(
    experiment,
    on="client_id",
    how="inner"
)

clean_data = web.merge(
    client_data,
    on="client_id",
    how="inner"
)

print("Client data shape:", client_data.shape)
print("Clean data shape:", clean_data.shape)

clean_data.head()


## Validate A/B test assignment

In [ ]:

variation_check = clean_data.groupby("client_id")["variation"].nunique()

display(variation_check.value_counts())

invalid_clients = variation_check[variation_check > 1].index

print("Invalid clients:", len(invalid_clients))

clean_data = clean_data[~clean_data["client_id"].isin(invalid_clients)]


## Create session-level dataset

In [ ]:

session_data = (
    clean_data
    .groupby(["variation", "client_id", "visitor_id", "visit_id"], as_index=False)
    .agg(
        max_step=("step_num", "max"),
        first_timestamp=("date_time", "min"),
        last_timestamp=("date_time", "max"),
        number_of_steps=("process_step", "count")
    )
)

session_data["completed"] = session_data["max_step"] == 4

session_data["session_duration_seconds"] = (
    session_data["last_timestamp"] - session_data["first_timestamp"]
).dt.total_seconds()

session_data.head()


## Merge session features back into clean data

In [ ]:

clean_data = clean_data.merge(
    session_data[
        [
            "client_id",
            "visitor_id",
            "visit_id",
            "completed",
            "session_duration_seconds",
            "max_step"
        ]
    ],
    on=["client_id", "visitor_id", "visit_id"],
    how="left"
)

clean_data.head()


## Final checks

In [ ]:

print("Final clean data shape:", clean_data.shape)
print("Final session data shape:", session_data.shape)

display(clean_data["variation"].value_counts())
display(clean_data["process_step"].value_counts())
display(clean_data.isna().sum())


## Export clean data

In [ ]:

clean_data.to_csv(f"{PROCESSED_DATA_PATH}/clean_vanguard_data.csv", index=False)
session_data.to_csv(f"{PROCESSED_DATA_PATH}/session_vanguard_data.csv", index=False)

print("Saved clean_vanguard_data.csv")
print("Saved session_vanguard_data.csv")
